# **SpongeWorks**


## Setup 

In [ ]:
from pyrealm.splash.splash import SplashModel
from pyrealm.core.calendar import Calendar
import pandas as pd
import numpy as np
import xarray as xr
from rasterio import features
import rioxarray
import hvplot.xarray
from minio import Minio
from io import BytesIO
from collections import defaultdict
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import Point, box
import contextily as ctx
import openeo
from pyproj import Transformer
import geoviews as gv
import hvplot.pandas
import geoviews.tile_sources as gts
import seaborn as sns
import panel as pn
import requests as r
from httpx import Client
from glob import glob
from matplotlib.cm import viridis, plasma, RdYlBu_r
from bokeh.models import ColumnDataSource, TapTool, CustomJS
from bokeh.plotting import figure, show, output_notebook
output_notebook()
import os 
from pyrealm import splash
import warnings
warnings.filterwarnings("ignore")

In [ ]:
def assign_lon_lat_to_xarray(data, lon, lat):
    """
    Assigns longitude and latitude values to xarray's x and y coordinates.

    Args:
        lai_data (xr.Dataset): The xarray Dataset.
        lon (np.ndarray): 2D array of longitude values.
        lat (np.ndarray): 2D array of latitude values.

    Returns:
        xr.Dataset: The xarray Dataset with updated coordinates.
    """

    # Ensure lon and lat match the shape of the x and y dimensions.
    if lon.shape != (data.y.size, data.x.size) or lat.shape != (data.y.size, data.x.size):
        raise ValueError("lon and lat arrays do not match the shape of xarray's x and y dimensions.")

    # Extract 1D arrays to assign to x and y coordinates.
    new_lon_x = lon[0, :]  # First row of lon (representing x)
    new_lat_y = lat[:, 0]  # First column of lat (representing y)

    # Assign the new coordinates:
    data = data.assign_coords({"x": new_lon_x, "y": new_lat_y})

    return data

In [ ]:
site = 'leze'  # Options: 'wallingford', 'MuW', 'leze'
aoi = gpd.read_file("/Users/vm/Desktop/map.geojson")
aoi = aoi.set_crs("EPSG:4326", allow_override=True)
W, S, E, N = aoi.total_bounds
# aoi.plot()

## Source and process raw data
------
This section focuses on sourcing and processing the necessary datasets for our analysis, including meteorological data, Sentinel-1 SAR imagery, and Sentinel-2 optical imagery.

### ERA5 Met Data

In [ ]:
# ## Produce met_DF dataset
# folders_met = []
# for file in glob("/Users/vm/Documents/Scripts and Data/data/ERA5/*") : 
# 	folders_met.append(file)
# folders_met.sort()

# siteLat = aoi.geometry.centroid.y[0] #.centroid.y.iloc[0]
# siteLon = aoi.geometry.centroid.x[0] #.centroid.x.iloc[0]

# met_DF = pd.DataFrame(columns=['date','minT','maxT','srad','vpd','wind','photopd','21d_vpd','21d_minT','21d_photopd','prec'])

# for ii in range(len(folders_met)):

# 	ds1 = xr.open_dataset(f"{folders_met[ii]}/data_stream-oper_stepType-instant.nc",engine='netcdf4')
# 	ds2 = xr.open_dataset(f"{folders_met[ii]}/data_stream-oper_stepType-accum.nc",engine='netcdf4')
# 	dsloc1 = ds1.sel(longitude=float(siteLon),latitude=float(siteLat),method='nearest') 
# 	dsloc2 = ds2.sel(longitude=float(siteLon),latitude=float(siteLat),method='nearest') 
# 	df1 = dsloc1.to_dataframe()
# 	df2 = dsloc2.to_dataframe()
# 	## unit conversion
# 	df1.t2m = df1.t2m - 273.15
# 	df1.d2m = df1.d2m - 273.15
# 	## relative humidity (http:/andrew.rsmas.miami.edu/bmcnoldy/Humidity.html)
# 	df1['RH'] = 100*(np.exp((17.625*df1.d2m)/(243.04+df1.d2m))/np.exp((17.625*df1.t2m)/(243.04+df1.t2m))) 
# 	## vapor pressure deficit (http:/cronklab.wikidot.com/calculation-of-vapour-pressure-deficit)
# 	df1['VPD'] = (1-(df1.RH/100)) * (610.7*10**(7.5*df1.t2m/(237.3+df1.t2m)))
# 	df1['windspeed'] = (df1.u10**2 + df1.v10**2)**0.5

# 	df1 = df1.reset_index()
# 	df1.index = pd.to_datetime(df1.valid_time)
# 	df1 = df1.drop(columns=['expver'])
# 	df2 = df2.reset_index()
# 	df2.index = pd.to_datetime(df2.valid_time)
# 	df2 = df2.drop(columns=['expver'])

# 	for t in range(len(df1.resample('D').mean())):
# 			met_DF = met_DF._append({'date' : (df1.resample('D').mean()).index[t],
# 									'doy'  : (df1.resample('D').mean()).index.dayofyear[t],
# 									'avgT' : (df1.t2m.resample('D',label='left').mean()).iloc[t],
# 									'prec' : (df2.tp.resample('D',label='left').mean()).iloc[t] * 1000, # / 3600 , # m to mm.s-1 i.e. kg.rain.m-2.s-1
# 									'wind' : (df1.windspeed.resample('D',label='left').mean()).iloc[t],
# 									'vpd'  : (df1.VPD.resample('D',label='left').mean()).iloc[t],
# 									},ignore_index=True)



### Sentinel-1 SAR and Sentinel-2 Optical Data

In [ ]:
# # Connect to the back-end and authenticate. 
# connection = openeo.connect("openeofed.dataspace.copernicus.eu")
# connection.authenticate_oidc()

# # Set the period of interest 
# poi = ["2017-01-01", "2024-12-31"]

# # Initialize our datacube object with the area of interest 
# # and the time range of interest using Sentinel 2 data.
# datacube = connection.load_collection(
#     "SENTINEL2_L2A",
#     spatial_extent={"west": W, "south": S, "east": E, "north": N}, # "crs": "EPSG:4326"
#     temporal_extent=poi,
#     bands=["B03", "B08", "SCL"],
#     max_cloud_cover=30,
# )

# # From this data cube, we can now select the individual bands and rescale the digital number values to physical reflectances:
# B03 = datacube.band("B03") * 0.0001
# B08 = datacube.band("B08") * 0.0001
# SCL = datacube.band("SCL") # Scene classification layer

# # Cloud masking using the SCL band
# cloud_mask = ((SCL == 8) | (SCL == 9) | (SCL == 3)) * 1.0

# # Apply the cloud mask
# masked_cube = datacube.mask(cloud_mask)

# # Recalculate the bands after masking
# B03_masked = masked_cube.band("B03") * 0.0001
# B08_masked = masked_cube.band("B08") * 0.0001

# # Calculate sar
# sar_cube = (B03_masked - B08_masked) / (B03_masked + B08_masked)

# sar_cube.download('/Users/vm/Desktop/dir/spongeworks/data/wallingford/ndwi_2017_2024.nc', format="NetCDF")

# # Query the Sentinel-1 data collection for the same period
# datacube = connection.load_collection(
#     "SENTINEL1_GRD",
#     spatial_extent={"west": W, "south": S, "east": E, "north": N}, # "crs": "EPSG:4326"
#     temporal_extent=poi,
#     bands=["VV", "VH"]
# )

# # Calculate backscatter using the datacube
# s1_scatter = datacube.sar_backscatter(coefficient="sigma0-ellipsoid", elevation_model="COPERNICUS_30", noise_removal=True)
# s1bs = s1_scatter.apply(lambda x: 10 * x.log(base=10))

# # Save to netcdf
# s1bs.download("/Users/vm/Desktop/dir/spongeworks/data/wallingford/sar_bs_2017_2024.nc")


### Copernicus Land Monitoring Service - Surface Soil Moisture (SSM) - 1km resolution

In [ ]:
# Ensure site directory exists
site_dir = f"/Users/vm/Desktop/dir/spongeworks/data/{site}/ssm_2015_2025"
os.makedirs(site_dir, exist_ok=True)

# Connect to the back-end and authenticate. 
connection = openeo.connect("openeofed.dataspace.copernicus.eu")
connection.authenticate_oidc()

# Set the period of interest 
poi = ["2015-01-01", "2025-12-30"]

# Initialize our datacube object with the area of interest 
datacube = connection.load_collection(
    "CGLS_SSM_V1_EUROPE",
    spatial_extent={"west": W, "south": S, "east": E, "north": N}, # "crs": "EPSG:4326"
    temporal_extent=poi,
    bands=["ssm"])

job = datacube.create_job(title=site)
job.start_and_wait()
job.get_results().download_files(site_dir)

tif_files = sorted(glob(f"{site_dir}/*.tif"))

if not tif_files:
    raise FileNotFoundError("No .tif files found in the target folder.")

# Read each tif as DataArray, drop singleton band dimension if present
ssm_list = [rioxarray.open_rasterio(fp).squeeze(drop=True) for fp in tif_files]

# Extract date from filename using characters between -15 and -5
date_strs = [fp.split("/")[-1][-15:-5] for fp in tif_files]
time_values = pd.to_datetime(date_strs, format="%Y-%m-%d", errors="coerce")

if time_values.isna().any():
    bad = [tif_files[i] for i, v in enumerate(time_values.isna()) if v][:5]
    raise ValueError(f"Could not parse dates from some filenames. Examples: {bad}")

# Concatenate along time
ssm_merged = xr.concat(ssm_list, dim="time").assign_coords(time=time_values)

# Convert to Dataset and save
ssm_ds = ssm_merged.to_dataset(name="ssm")
out_nc = f"/Users/vm/Desktop/dir/spongeworks/data/{site}/ssm_2020_2025.nc"
ssm_ds.to_netcdf(out_nc)


### DEM 

In [ ]:
# Ensure site directory exists
site_dir = f"/Users/vm/Desktop/dir/spongeworks/data/{site}/ssm_2015_2025"
os.makedirs(site_dir, exist_ok=True)

# Connect to the back-end and authenticate. 
connection = openeo.connect("openeofed.dataspace.copernicus.eu")
connection.authenticate_oidc()

# Load the Copernicus DEM 30m collection (or use another DEM if needed)
dem_cube = connection.load_collection(
    "COPERNICUS_30",
    spatial_extent={"west": W, "south": S, "east": E, "north": N, "crs": "EPSG:4326"},
    bands=["DEM"]
)

# Download the DEM as a NetCDF file
dem_cube.download(f"/Users/vm/Desktop/dir/spongeworks/data/{site}/dem_copernicus_30m.nc", format="NetCDF")

## Run SPLASH 
------ 
* documentation : https://pyrealm.readthedocs.io/en/latest/users/splash.html
* INPUT DATA : daily time series of precipitation, temperature and solar fraction (1 - cloud cover) 
* OUTPUT DATA : soil moisture, actual evapotranspiration (AET) and surface water runoff 

In [ ]:
def get_elevation(lat, lon):
    """
    Query Open-Elevation API to get elevation (in meters) for a given latitude and longitude.
    Returns elevation in meters, or None if not found.
    """
    url = "https://api.open-elevation.com/api/v1/lookup"
    params = {"locations": f"{lat},{lon}"}
    try:
        response = r.get(url, params=params)
        response.raise_for_status()
        results = response.json().get("results", [])
        if results:
            return results[0]["elevation"]
    except Exception as e:
        print(f"Error fetching elevation: {e}")
    return None


def daylength(dayOfYear, lat):
    """Computes the length of the day (the time between sunrise and
    sunset) given the day of the year and latitude of the location.
    Accepts dayOfYear as a scalar or numpy array.
    Uses the Brock model for the computations.

    Parameters
    ----------
    dayOfYear : int or np.ndarray
        The day of the year. 1 corresponds to 1st of January
        and 365 to 31st December (on a non-leap year).
    lat : float
        Latitude of the location in degrees. Positive values
        for north and negative for south.

    Returns
    -------
    d : float or np.ndarray
        Daylength in hours.
    """
    dayOfYear = np.asarray(dayOfYear)
    latInRad = np.deg2rad(lat)
    declinationOfEarth = 23.45 * np.sin(np.deg2rad(360.0 * (283.0 + dayOfYear) / 365.0))
    tan_lat = np.tan(latInRad)
    tan_dec = np.tan(np.deg2rad(declinationOfEarth))
    x = -tan_lat * tan_dec

    # Use numpy for vectorized computation
    d = np.where(
        x <= -1.0, 24.0,
        np.where(
            x >= 1.0, 0.0,
            2.0 * np.rad2deg(np.arccos(x)) / 15.0
        )
    )
    return d

### Load met data from minio bucket 

In [ ]:

# MinIO connection details
minio_endpoint = "general-gensto.datalabs.ceh.ac.uk"
minio_bucket = "europemetsoildata"
minio_folder = "metdata/Europe/"
minio_secure = True

# Create MinIO client (anonymous access)
client = Minio(
    minio_endpoint,
    access_key="cb17a3b6-65e8-4cb1-8b8c-8f51f513f23a",
    secret_key="af8d4045-ce18-41c6-871f-a4394eb58c07",
    secure=minio_secure
)

# # List all objects in the folder
objects = client.list_objects(minio_bucket, prefix=minio_folder, recursive=True)
# # Filter for NetCDF files in the folder
nc_file_list = [obj.object_name for obj in objects if obj.object_name.endswith('.nc')]

# Organize files by variable
files_by_variable = defaultdict(list)
for fname in nc_file_list:
    # Extract variable name (assumes format: YEAR_variablename.nc)
    var = fname.split('_', 1)[1].rsplit('.', 1)[0]
    files_by_variable[var].append(fname)

# Sort files for each variable by year (optional, for consistency)
for var in files_by_variable:
    files_by_variable[var].sort()

# files_by_variable now maps variable names to lists of yearly NetCDF files
# Example: print the mapping
for var, files in files_by_variable.items():
    print(f"{var}: {files}")

# Keep only selected variables in files_by_variable
vars_to_keep = ['2m_temperature', 'total_cloud_cover', 'total_precipitation']
files_by_variable = {k: v for k, v in files_by_variable.items() if k in vars_to_keep}

# Keep only files from years 2020 to 2025 for each variable
def is_year_in_range(fname, start=2015, end=2025):
    # Assumes filename format: .../YYYY_variablename.nc
    try:
        year = int(fname.split('/')[-1].split('_')[0])
        return start <= year <= end
    except Exception:
        return False

for var in files_by_variable:
    files_by_variable[var] = [f for f in files_by_variable[var] if is_year_in_range(f)]

# Dictionary to hold merged datasets per variable
datasets_by_variable = {}

for variable, files in files_by_variable.items():
    # Open multiple NetCDF files as a single xarray dataset
    file_objs = [BytesIO(client.get_object(minio_bucket, fname).read()) for fname in files]
    ds = xr.open_mfdataset(file_objs, combine='by_coords')
    datasets_by_variable[variable] = ds

print(datasets_by_variable.keys())

In [ ]:
temp_hourly = datasets_by_variable['2m_temperature']           
temp_daily = temp_hourly.resample(valid_time='1D').mean()
temp_daily['t2m'] = temp_daily['t2m'] - 273.15
temp_daily = temp_daily.rio.write_crs("EPSG:4326")
prec_hourly = datasets_by_variable['total_precipitation']
# Convert precipitation from meters to millimeters
prec_daily = prec_hourly.resample(valid_time='1D').sum()
prec_daily['tp'] = prec_daily['tp'] * 1000  # m to mm
prec_daily = prec_daily.rio.write_crs("EPSG:4326")
cloud_hourly = datasets_by_variable['total_cloud_cover']

In [ ]:
del datasets_by_variable

In [ ]:
# Choose your latitude of interest for each pixel
lats = cloud_hourly['latitude'].values
lons = cloud_hourly['longitude'].values

# Get daily dates from the available data
daily_dates = pd.to_datetime(cloud_hourly['valid_time'].values).normalize().unique()
daily_dates = pd.DatetimeIndex(daily_dates)

# Extract total cloud cover at 8, 12, and 16 hours
tcc_sel = cloud_hourly.sel(valid_time=cloud_hourly['valid_time'].dt.hour.isin([8, 12, 16]))

# Compute daily mean for each pixel
tcc_daily = tcc_sel.resample(valid_time='1D').mean(dim='valid_time')
tcc_daily = tcc_daily.sel(valid_time=daily_dates)

# Use xarray.apply_ufunc to compute daylength for each latitude and day
def daylength_vec(dayofyear, lat):
    return np.array([daylength(dayofyear, float(la)) for la in lat]).T

dl_xr = xr.apply_ufunc(
    daylength_vec,
    tcc_daily['valid_time'].dt.dayofyear,
    tcc_daily['latitude'],
    input_core_dims=[['valid_time'], ['latitude']],
    output_core_dims=[['valid_time', 'latitude']],
    vectorize=True,
    dask='parallelized',
    output_dtypes=[float]
)

# Broadcast daylength to match tcc dimensions (time, latitude, longitude)
dl_broadcast = dl_xr.expand_dims(longitude=tcc_daily['longitude'])

# Calculate sunshine fraction: sf = daylength * (1 - tcc)
sf_xr = dl_broadcast * (1 - tcc_daily['tcc'])

# Optionally normalize per-pixel (time, lat, lon)
sf_min = sf_xr.min(dim='valid_time')
sf_max = sf_xr.max(dim='valid_time')
# Align sf_min and sf_max to sf_xr's dimensions for broadcasting
sf_min_aligned = sf_min.transpose('longitude', 'latitude').expand_dims(valid_time=sf_xr.valid_time)
sf_max_aligned = sf_max.transpose('longitude', 'latitude').expand_dims(valid_time=sf_xr.valid_time)
sf_min_aligned = sf_min_aligned.transpose('longitude', 'valid_time', 'latitude')
sf_max_aligned = sf_max_aligned.transpose('longitude', 'valid_time', 'latitude')

sf_norm = (sf_xr - sf_min_aligned) / (sf_max_aligned - sf_min_aligned)
sf_norm = sf_norm.where(~np.isclose(sf_max_aligned, sf_min_aligned), 0)

sf_norm.name = 'sf'
sf_norm = sf_norm.transpose('valid_time', 'latitude', 'longitude')
sf_norm = sf_norm.rio.write_crs("EPSG:4326")

In [ ]:
# Merge sf_daily, prec_daily, and temp_daily into a single xarray dataset
merged_ds = xr.merge([sf_norm, prec_daily, temp_daily])
merged_ds = merged_ds.transpose('valid_time','latitude', 'longitude')
merged_ds = merged_ds.rio.clip(geometries=aoi.geometry, all_touched=True, drop=True, invert=False)
merged_ds 

In [ ]:
dem = xr.open_dataset(f"/Users/vm/Desktop/dir/spongeworks/data/{site}/dem_copernicus_30m.nc")['DEM']
# Merge DEM data along the 't' dimension by filling missing values in one entry with values from the other
if "t" in dem.dims and dem.sizes["t"] > 1:
    # Use where to fill missing values in t=0 with t=1 and vice versa, then drop the t dimension
    dem_merged = dem.isel(t=0).where(~np.isnan(dem.isel(t=0)), dem.isel(t=1))
    dem_merged = dem_merged.where(~np.isnan(dem_merged), dem.isel(t=1).where(~np.isnan(dem.isel(t=1)), dem.isel(t=0)))
    dem = dem_merged
    
# Set CRS to EPSG:4326 if not already set
if not dem.rio.crs:
    dem = dem.rio.write_crs("EPSG:4326")

dem_clipped = dem.rio.clip(geometries=aoi.geometry, all_touched=True, drop=True, invert=False)
dem_resampled = dem_clipped.interp(y=merged_ds.latitude, x=merged_ds.longitude, method='nearest', 
                                   kwargs={"fill_value": "extrapolate"})

# Correct shape of elevatio and latitude arrays to match merged_ds dimensions
elv = np.broadcast_to(dem_resampled.values, merged_ds.t2m.shape)
lat = np.broadcast_to(merged_ds.latitude.values[None, :, None], merged_ds.t2m.shape)


In [43]:
print("lat shape:", lat.shape)
print("elv shape:", elv.shape)
print("dates shape:", merged_ds.valid_time.to_numpy().shape)
print("sf shape:", merged_ds.sf.to_numpy().shape)
print("tc shape:", merged_ds.t2m.to_numpy().shape)
print("pn shape:", merged_ds.tp.to_numpy().shape)

sf shape: (4018, 3, 2)
tc shape: (4018, 3, 2)
pn shape: (4018, 3, 2)


In [44]:
splash = SplashModel(
    lat=lat,
    elv=elv,
    dates=Calendar(merged_ds.valid_time.to_numpy()),
    sf=merged_ds.sf.to_numpy(),
    tc=merged_ds.t2m.to_numpy(),
    pn=merged_ds.tp.to_numpy(),
)

In [ ]:
# merged_ds.sf[:, 0, 0].hvplot.line(x='valid_time', y='sf', title='Sunshine Fraction at First Pixel')
# merged_ds.t2m[:, 0, 0].hvplot.line(x='valid_time', y='t2m', title='Temperature at First Pixel')

In [ ]:
init_soil_moisture = splash.estimate_initial_soil_moisture(verbose=True)
# aet, wn, ro = splash.estimate_daily_water_balance(init_soil_moisture, day_idx=0)
aet_out, wn_out, ro_out = splash.calculate_soil_moisture(init_soil_moisture)

In [ ]:
splash.calculate_soil_moisture?

In [ ]:
sm = xr.open_dataset(f"/Users/vm/Desktop/dir/spongeworks/data/{site}/ssm_2020_2025.nc")

In [ ]:
# Resample sm.ssm (time, y, x) to match merged_ds (valid_time, latitude, longitude)
# We'll use xarray's interp to interpolate both spatially and temporally

# First, rename sm dimensions to match merged_ds
sm_interp = sm.ssm.rename({'time': 'valid_time', 'y': 'latitude', 'x': 'longitude'})

# Interpolate to merged_ds's grid (both time and space)
sm_resampled = sm_interp.interp(
    valid_time=merged_ds.valid_time,
    latitude=merged_ds.latitude,
    longitude=merged_ds.longitude,
    method="linear"
)

# sm_resampled now has the same dimensions as merged_ds
sm_resampled

In [ ]:
sm_mean = sm.ssm.mean(dim=["x", "y"])
# Convert to DataFrame with datetime index
sm_mean_df = sm_mean.to_dataframe(name='soil_moisture')
sm_mean_df.index.name = 'date'
# Min-max normalization of soil moisture values
sm_min = sm_mean_df['soil_moisture'].min()
sm_max = sm_mean_df['soil_moisture'].max()
sm_mean_df['soil_moisture_norm'] = (sm_mean_df['soil_moisture'] - sm_min) / (sm_max - sm_min)
sm_mean_df.soil_moisture_norm.plot()

In [ ]:
wn_out_mean = np.nanmean(wn_out, axis=(1, 2))
wn_out_mean_df = pd.DataFrame({'soil_moisture': wn_out_mean}, index=merged_ds.valid_time.values)
wn_out_mean_df.index.name = 'date'
# Min-max normalization of soil moisture values
sm_min = wn_out_mean_df['soil_moisture'].min()
sm_max = wn_out_mean_df['soil_moisture'].max()
wn_out_mean_df['soil_moisture_norm'] = (wn_out_mean_df['soil_moisture'] - sm_min) / (sm_max - sm_min)
wn_out_mean_df.soil_moisture_norm.plot()

In [ ]:
# Merge sm_mean_df and wn_out_mean_df on their datetime index
merged_soil_moisture = sm_mean_df.merge(wn_out_mean_df, left_index=True, right_index=True, how='outer', suffixes=('_ssm', '_splash'))
merged_soil_moisture

In [ ]:
import seaborn as sns 
sns.scatterplot(
    data=merged_soil_moisture.dropna(),
    x="soil_moisture_norm_ssm",
    y="soil_moisture_norm_splash"
)